In [11]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import joblib

# **Feature Engineering**

In [12]:
df = pd.read_csv('data/train.csv')
df.drop(columns=['Unnamed: 0', 'id'], inplace=True, errors='ignore')

In [13]:
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(0)

In [14]:
df['Total_Delay'] = df['Departure Delay in Minutes'] + df['Arrival Delay in Minutes']
df = df.drop(['Departure Delay in Minutes', 'Arrival Delay in Minutes'], axis=1)

I fill NaN with 0 and combined both features related to delays because of strong correlation between them.

## Creating piplines

In [15]:
cat_features = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [16]:
num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

### Split and preprocessing on dataset

In [18]:
X = df.drop('satisfaction', axis=1)
y = df['satisfaction'].map({'neutral or dissatisfied': 0, 'satisfied': 1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

### Safe dataset after Feature Engineering

In [20]:
joblib.dump((X_train, X_test, y_train, y_test), 'data/processed_data.pkl')
joblib.dump(preprocessor, 'models/preprocessor.pkl')

['models/preprocessor.pkl']